In [0]:
# ==============================================================================
# CAMADA GOLD: Modelagem Dimensional (Star Schema) & Metadados
# ==============================================================================

from pyspark.sql.functions import col, dense_rank, current_timestamp
from pyspark.sql.window import Window

# 1. Leitura da Tabela Delta Silver
df_silver = spark.table("workspace.default.silver_listings")

print("Iniciando a criação do Star Schema na Camada Gold...")

# ------------------------------------------------------------------------------
# DIMENSÃO ANFITRIÃO (dim_host)
# ------------------------------------------------------------------------------
df_dim_host = df_silver.select(
    "host_id",
    "host_name",
    "host_since",
    "is_superhost",
    "host_listings_count"
).distinct()

df_dim_host.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.dim_host")

# ------------------------------------------------------------------------------
# DIMENSÃO LOCALIZAÇÃO (dim_location)
# ------------------------------------------------------------------------------
# Gerando ID único para cada combinação de Bairro e Zona
window_loc = Window.orderBy("neighbourhood", "zone")
df_dim_location = df_silver.select("neighbourhood", "zone").distinct() \
    .withColumn("location_id", dense_rank().over(window_loc)) \
    .select("location_id", "neighbourhood", "zone")

df_dim_location.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.dim_location")

# ------------------------------------------------------------------------------
# DIMENSÃO IMÓVEL E COMODIDADES (dim_property)
# ------------------------------------------------------------------------------
df_dim_property = df_silver.select(
    col("listing_id").alias("property_id"),
    "property_type",
    "room_type",
    "accommodates",
    "bedrooms",
    "beds",
    "has_wifi",
    "has_air_conditioning",
    "has_pool",
    "has_sea_view"
).distinct()

df_dim_property.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.dim_property")

# ------------------------------------------------------------------------------
# TABELA FATO (fact_listings)
# ------------------------------------------------------------------------------
# Cruzamento com dim_location para recuperar a Foreign Key location_id
df_fact = df_silver.join(df_dim_location, on=["neighbourhood", "zone"], how="inner") \
    .select(
        col("listing_id").alias("property_id"),
        "host_id",
        "location_id",
        "price",
        "minimum_nights",
        "maximum_nights",
        "number_of_reviews",
        "review_score",
        col("latitude"),
        col("longitude"),
        current_timestamp().alias("_created_at")
    )

df_fact.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.fact_listings")

print("✅ Tabelas da Camada Gold criadas com sucesso!")

In [0]:
%sql
-- ==============================================================================
-- DOCUMENTAÇÃO DO CATÁLOGO DE DADOS (UNITY CATALOG)
-- ==============================================================================

-- 1. DIMENSÃO ANFITRIÃO
COMMENT ON TABLE workspace.default.dim_host IS 'Dimensão com o perfil e métricas operacionais/reputacionais dos anfitriões.';
COMMENT ON COLUMN workspace.default.dim_host.host_id IS 'Chave Primária (PK). Identificador único do anfitrião no Airbnb.';
COMMENT ON COLUMN workspace.default.dim_host.host_name IS 'Nome cadastrado pelo anfitrião.';
COMMENT ON COLUMN workspace.default.dim_host.host_since IS 'Data de cadastro do anfitrião na plataforma.';
COMMENT ON COLUMN workspace.default.dim_host.is_superhost IS 'Indicador booleano (True/False) se possui o selo Superhost de excelência.';
COMMENT ON COLUMN workspace.default.dim_host.host_listings_count IS 'Quantidade total de imóveis gerenciados pelo anfitrião.';

-- 2. DIMENSÃO LOCALIZAÇÃO
COMMENT ON TABLE workspace.default.dim_location IS 'Dimensão geográfica que consolida bairros e zonas do Rio de Janeiro.';
COMMENT ON COLUMN workspace.default.dim_location.location_id IS 'Chave Primária (PK). Identificador único sintético da localização.';
COMMENT ON COLUMN workspace.default.dim_location.neighbourhood IS 'Nome oficial do bairro no município do Rio de Janeiro.';
COMMENT ON COLUMN workspace.default.dim_location.zone IS 'Mapeamento regional do bairro (Zona Sul, Zona Norte, Zona Oeste, Centro, Outros).';

-- 3. DIMENSÃO IMÓVEL E COMODIDADES
COMMENT ON TABLE workspace.default.dim_property IS 'Dimensão estrutural com tipologia e atributos qualitativos do imóvel.';
COMMENT ON COLUMN workspace.default.dim_property.property_id IS 'Chave Primária (PK). Identificador único do anúncio do imóvel.';
COMMENT ON COLUMN workspace.default.dim_property.property_type IS 'Categoria do imóvel (ex: Casa, Apartamento, Quarto de Hotel).';
COMMENT ON COLUMN workspace.default.dim_property.room_type IS 'Tipo de acomodação (Espaço inteiro, Quarto inteiro, Quarto compartilhado).';
COMMENT ON COLUMN workspace.default.dim_property.accommodates IS 'Capacidade máxima de hóspedes.';
COMMENT ON COLUMN workspace.default.dim_property.bedrooms IS 'Quantidade de quartos disponíveis.';
COMMENT ON COLUMN workspace.default.dim_property.beds IS 'Quantidade de camas disponíveis.';
COMMENT ON COLUMN workspace.default.dim_property.has_wifi IS 'Indicador booleano (True/False) de presença de Wi-Fi.';
COMMENT ON COLUMN workspace.default.dim_property.has_air_conditioning IS 'Indicador booleano (True/False) de presença de Ar-Condicionado.';
COMMENT ON COLUMN workspace.default.dim_property.has_pool IS 'Indicador booleano (True/False) de presença de Piscina.';
COMMENT ON COLUMN workspace.default.dim_property.has_sea_view IS 'Indicador booleano (True/False) de presença de Vista para o Mar/Oceano.';

-- 4. TABELA FATO
COMMENT ON TABLE workspace.default.fact_listings IS 'Tabela Fato centralizada com as métricas transacionais e regras dos anúncios.';
COMMENT ON COLUMN workspace.default.fact_listings.property_id IS 'Chave Estrangeira (FK). Aponta para dim_property.property_id.';
COMMENT ON COLUMN workspace.default.fact_listings.host_id IS 'Chave Estrangeira (FK). Aponta para dim_host.host_id.';
COMMENT ON COLUMN workspace.default.fact_listings.location_id IS 'Chave Estrangeira (FK). Aponta para dim_location.location_id.';
COMMENT ON COLUMN workspace.default.fact_listings.price IS 'Valor monetário da diária em Reais (BRL).';
COMMENT ON COLUMN workspace.default.fact_listings.minimum_nights IS 'Número mínimo de noites exigido para reserva.';
COMMENT ON COLUMN workspace.default.fact_listings.maximum_nights IS 'Número máximo de noites permitido para reserva.';
COMMENT ON COLUMN workspace.default.fact_listings.number_of_reviews IS 'Total de avaliações recebidas pelo anúncio.';
COMMENT ON COLUMN workspace.default.fact_listings.review_score IS 'Nota média geral dada pelos hóspedes (0 a 5).';
COMMENT ON COLUMN workspace.default.fact_listings.latitude IS 'Coordenada geográfica de latitude.';
COMMENT ON COLUMN workspace.default.fact_listings.longitude IS 'Coordenada geográfica de longitude.';
COMMENT ON COLUMN workspace.default.fact_listings._created_at IS 'Data/Hora de inserção do registro no Data Lakehouse.';